<a href="https://colab.research.google.com/github/pranjalbainsla/ml-notes/blob/master/micrograd_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# micrograd exercises

1. watch the [micrograd video](https://www.youtube.com/watch?v=VMj-3S1tku0) on YouTube
2. come back and complete these exercises to level up :)

## section 1: derivatives

In [ ]:
# here is a mathematical expression that takes 3 inputs and produces one output
from math import sin, cos

def f(a, b, c):
  return -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5

print(f(2, 3, 4))

In [ ]:
# write the function df that returns the analytical gradient of f
# i.e. use your skills from calculus to take the derivative, then implement the formula
# if you do not calculus then feel free to ask wolframalpha, e.g.:
# https://www.wolframalpha.com/input?i=d%2Fda%28sin%283*a%29%29%29

def gradf(a, b, c):
  fa = -3*a**2 - 0.5*a**-0.5
  fb = 3*cos(3*b) + 2.5*b**1.5
  fc = 1/c**2
  return [fa, fb, fc]

# expected answer is the list of
ans = [-12.353553390593273, 10.25699027111255, 0.0625]
yours = gradf(2, 3, 4)
for dim in range(3):
  ok = 'OK' if abs(yours[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {yours[dim]}")


In [ ]:
# now estimate the gradient numerically without any calculus, using
# the approximation we used in the video.
# you should not call the function df from the last cell

# -----------
a = 2
b = 3
c = 4
h = 0.000001 # 0.00001 wasn't good enough

original = f(a, b, c)
fa = (f(a+h, b, c) - original)/h
fb = (f(a, b+h, c) - original)/h
fc = (f(a, b, c+h) - original)/h
numerical_grad = [fa, fb, fc]
# -----------

for dim in range(3):
  ok = 'OK' if abs(numerical_grad[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad[dim]}")


In [ ]:
# there is an alternative formula that provides a much better numerical
# approximation to the derivative of a function.
# learn about it here: https://en.wikipedia.org/wiki/Symmetric_derivative
# implement it. confirm that for the same step size h this version gives a
# better approximation.

# -----------
h2 = 0.001

fa2 = (f(a+h2, b, c) - f(a-h2, b, c))/(2*h2)
fb2 = (f(a, b+h2, c) - f(a, b-h2, c))/(2*h2)
fc2 = (f(a, b, c+h2) - f(a, b, c-h2))/(2*h2)
numerical_grad2 = [fa2, fb2, fc2]
# -----------

for dim in range(3):
  ok = 'OK' if abs(numerical_grad2[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad2[dim]}")


## section 2: support for softmax

In [ ]:
# Value class starter code, with many functions taken out
from math import exp, log

class Value:
  
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"
  
  def __add__(self, other): # exactly as in the video
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    
    return out
  
  # ------
  # re-implement all the other functions needed for the exercises below
  # your code here
  # TODO
  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    
    return out

  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data ** other, (self,), f'**{other}')

    def _backward():
      self.grad += other * (self.data**(other-1)) * out.grad
    out._backward = _backward

    return out
  
  def __truediv__(self, other):
    return self * other**-1

  def __radd__(self, other):
    return self + other
  
  def __rmul__(self, other):
    return self * other

  def __neg__(self): # -self
    return -1 * self

  def exp(self):
    x = self.data
    out = Value(exp(x), (self, ), 'exp')

    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward

    return out
  
  def log(self):
    x = self.data
    out = Value(log(x), (self, ), 'log')

    def _backward():
      self.grad += (1.0 * x**-1) * out.grad
    out._backward = _backward

    return out
  # ------

  def backward(self): # exactly as in video  
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()
 

In [ ]:
# without referencing our code/video __too__ much, make this cell work
# you'll have to implement (in some cases re-implemented) a number of functions
# of the Value object, similar to what we've seen in the video.
# instead of the squared error loss this implements the negative log likelihood
# loss, which is very often used in classification.

# this is the softmax function
# https://en.wikipedia.org/wiki/Softmax_function
def softmax(logits):
  counts = [logit.exp() for logit in logits]
  denominator = sum(counts)
  out = [c / denominator for c in counts]
  return out

# this is the negative log likelihood loss function, pervasive in classification
logits = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]
probs = softmax(logits)
loss = -probs[3].log() # dim 3 acts as the label for this input example
loss.backward()
print(loss.data)

ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]
for dim in range(4):
  ok = 'OK' if abs(logits[dim].grad - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {logits[dim].grad}")


Logits, intuitively:

1. **Raw scores, not probabilities** — a model's last layer spits out one number per possible output class. These raw numbers are logits. They can be negative, positive, huge, tiny — no constraints.

2. **They encode "relative confidence"** — a higher logit means the model favors that option more, but the number itself isn't interpretable ("logit of 3.2" means nothing on its own).

3. **Softmax turns them into probabilities** — you exponentiate each logit and normalize by the sum. This squashes everything into a 0-1 range that sums to 1. So logits are basically "pre-probability" scores.

4. **Why not just output probabilities directly?** — logits are unbounded, which makes training math (gradients, loss functions like cross-entropy) cleaner and more numerically stable than working directly with squashed 0-1 values.

In [ ]:
# verify the gradient using the torch library
# torch should give you the exact same gradient
import torch
logits = torch.tensor([0.0, 3.0, -2.0, 1.0], requires_grad=True)
probs = torch.softmax(logits, dim=0)
loss = -probs[3].log()
loss.backward()

print(loss.item())

ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]
for dim in range(4):
  ok = 'OK' if abs(logits.grad[dim].item() - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {logits.grad[dim].item()}")



Softmax alternatives:

Sparsemax — instead of giving every option some tiny probability, it can assign exact zero to weak candidates, producing a genuinely sparse distribution. Useful when you want the model to be decisively "this or that," not "everything a little bit."

Sigmoid (per-logit) — used when classes aren't mutually exclusive (multi-label problems). Each logit gets its own independent 0-1 squashing via sigmoid, and they don't need to sum to 1, since more than one thing can be "true" at once.

Temperature-scaled softmax — technically still softmax, but dividing logits by a temperature T before exponentiating. Low T → sharper, more confident distribution; high T → flatter, more random. This is literally what "temperature" in LLM sampling controls.

Why softmax wins anyway — it's differentiable everywhere (good for gradient-based training), has a clean probabilistic interpretation, and pairs naturally with cross-entropy loss. Alternatives exist for special cases, but softmax is the default workhorse.